In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType, TimestampType

catalog_name = "ecommerce"

In [0]:
df = spark.table("ecommerce.bronze.brnz_brands")
df.show(10)

In [0]:
df_silver = df.withColumn('brand_name',F.trim(F.col('brand_name')))
df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn('brand_code', F.regexp_replace(F.col('brand_code'),r'[^A-Za-z0-9]',''))
df_silver.show(10)

In [0]:
df_silver.select("category_code").distinct().show(10)

In [0]:
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

df_silver = df_silver.replace(to_replace=anomalies, subset=["category_code"])
df_silver.select("category_code").distinct().show()

In [0]:
df_silver.show()
spark.sql("DROP TABLE IF EXISTS ecommerce.silver.slv_brands")
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable("ecommerce.silver.slv_brands")

In [0]:
df_bronze = spark.table("ecommerce.bronze.brz_category")

df_bronze.show(10)

In [0]:
df_duplicates = df_bronze.groupBy("category_code").count().filter("count > 1")
df_duplicates.show()
df_silver = df_bronze.dropDuplicates(["category_code"])
df_silver.show(10)

In [0]:
df_silver = df_silver.withColumn("category_code",F.upper(F.col("category_code")))
df_silver.show(10)

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable("ecommerce.silver.slv_category")


In [0]:
df_bronze = spark.table("ecommerce.bronze.brz_products")
#df_bronze.show(10)
row_count, columncount = df_bronze.count(), len(df_bronze.columns)
print(f"Row Count : {row_count}, Column Count : {columncount}")